# Molecular Devices FilterMax F5

```{device-card} molecular-devices-filtermax-f5
```

| Property | Value |
|---|---|
| PLR class | `FilterMaxF5` |
| Connection | RS-232 serial, automatic 38,400/9,600 baud detection, 7E1, no flow control |
| Current verified modes | Absorbance and luminescence |
| Current verified controls | Tray, filter slides, temperature, shaking, cancellation |

> **Validation status:** Direct PLR-to-F5 hardware validation succeeded. Full-plate absorbance at 450 nm agreed with the corresponding SoftMax Pro results; additional validated operations are recorded in the [protocol coverage](protocol-coverage.md). Fluorescence, FP, and TRF commands remain unavailable until they can be captured with a compatible excitation slide; the driver does not infer them.

## How it communicates

The F5 uses an ASTM-like framed serial protocol, not the SpectraMax `!COMMAND` protocol. PLR handles framing, checksums, acknowledgements, response chunking, device errors, and cancellation internally. Raw commands are intentionally not public.

## Physical setup

1. Connect the FilterMax F5 through its RS-232 cable or a USB-to-serial adapter.
2. Close SoftMax Pro and release the serial port. If Windows runs in VirtualBox with a host serial port mapping, save the results and shut Windows down normally; confirm the VM is powered off. Only one program may own the port.
3. Export your filter-slide definitions from SoftMax Pro as XML, or use the example catalog below when its definitions match your physical slides.
4. Keep the tray path clear. A plate is optional: inspection, tray operation, temperature control, and shaking run by default. Measurements are opt-in, with a guided plate-loading step before plate geometry.
5. Use a stable device path on Linux, such as `/dev/serial/by-id/...`.

For optional measurements, prepare a compatible plate with samples or blanks in the wells you intend to read, and install the appropriate filter slides. Excitation and emission slides occupy separate bays and are not interchangeable.

## Load the filter-slide catalog

Download {download}`filtermax_example_slides.xml <filtermax_example_slides.xml>` or use the copy alongside this notebook in the repository. It contains 16 filter definitions for excitation slides 1 and 2 and emission slide 1. Excitation slide 2 includes 260 nm at position 1 (12 nm bandwidth) and 450 nm at position 4 (8 nm bandwidth).

`FilterSlideCatalog.from_softmax_xml()` loads slide IDs, positions, wavelengths, bandwidths, and supported filter techniques. Measurement methods use this catalog to resolve a wavelength to a slide and position and to send that slide's filter definitions to the reader. Loading or resolving the catalog is local and does not contact the instrument.

Use your own SoftMax XML export for a different filter configuration. The driver checks installed slide IDs, but it cannot verify that the physical filters match the XML. Catalog entries for fluorescence, FP, and TRF do not enable those currently unsupported driver modes.

The next cell assumes this notebook's directory is the working directory. From the repository root, set `catalog_path` to `Path("docs/user_guide/molecular_devices/filtermax/filtermax_example_slides.xml")`. For your own export, use its local path instead.

In [ ]:
from pathlib import Path

from pylabrobot.molecular_devices.filtermax import FilterMaxF5, FilterSlideCatalog

catalog_path = Path("filtermax_example_slides.xml")
catalog = FilterSlideCatalog.from_softmax_xml(catalog_path)

## Check an example filter locally

Resolve the 450 nm absorbance filter to excitation slide 2, position 4. For 260 nm, use `wavelength=260` and `position=1`. This check does not move the reader or start a measurement.

In [ ]:
filter_selection = catalog.resolve(
  kind="excitation",
  technique="absorbance",
  wavelength=450,
  slide_id=2,
  position=4,
)
filter_selection

## Choose the serial port

List detected serial ports and identify the FilterMax F5 adapter. On Linux, prefer its stable `/dev/serial/by-id/...` path; Windows ports usually look like `COM3`. If no port is listed, connect the adapter and check its cable/permissions. Use the exact device path shown for the F5 below.

In [ ]:
from serial.tools import list_ports

[(port.device, port.description) for port in list_ports.comports()]

## Create the reader

Pass the loaded catalog as `filter_slide_catalog`. Creating the reader does not open the serial port. Connection and status queries can run without a catalog, but measurement methods require one. Enter the exact FilterMax F5 port from the list above when prompted.

In [ ]:
port = input("FilterMax serial port: ").strip()
if not port:
  raise ValueError("Enter the serial port for the FilterMax F5.")

reader = FilterMaxF5(
  name="filtermax-f5",
  port=port,
  filter_slide_catalog=catalog,
)

## Connect

`setup()` tries 38,400 baud first, then closes and reopens the port at 9,600 baud only if the initial handshake receives no response. It verifies the F5 identity and retains the working baud rate, which appears in the connection log. It does not change the reader's baud setting. Identity, protocol, and later transaction errors stop setup without retrying. All failed setup attempts release the serial port.

In [ ]:
await reader.setup()

## Inspect the instrument

Read identity, status, installed slide IDs, and temperature without changing instrument state.

In [ ]:
info = await reader.get_instrument_info()
status = await reader.get_status()
slides = await reader.get_filter_slides()
temperature = await reader.get_temperature()

info, status, slides, temperature

## Plate tray

Run the tray-out demonstration with the tray path clear. No plate is required; the optional measurement section below has its own loading step.

In [ ]:
await reader.move_plate_tray_out()

Confirm that your hands and any obstructions are clear before closing the tray. Enter `yes` to continue. If you decline, the notebook stops here with the tray open; clear the path and rerun this confirmation cell to continue.

In [ ]:
if input("Are your hands and the tray path clear? [y/N] ").strip().lower() not in ("y", "yes"):
  raise RuntimeError("Tray left open. Clear the path and rerun this confirmation cell to continue.")

Close the tray after confirming that its path is clear.

In [ ]:
await reader.move_plate_tray_in()

## Temperature control

Set a temperature in degrees Celsius, then query the current chamber temperature.

In [ ]:
await reader.set_temperature(25.0)
temperature = await reader.get_temperature()

Deactivate temperature control when it is no longer needed.

In [ ]:
await reader.deactivate_temperature_control()

## Shaking

Shaking is a finite-duration operation. Patterns are `"linear"` or `"orbital"`; speeds are `"low"`, `"medium"`, or `"high"`.

In [ ]:
await reader.start_shaking(pattern="linear", speed="medium", duration=5)

## Optional plate measurements

Press Enter at the first prompt to skip all measurements and continue to device errors and disconnect. Inspection, tray controls, temperature control, and shaking above do not depend on this choice.

If you choose measurements, select each example separately (`yes` to select; Enter to skip). Full-plate absorbance reads every well. All other examples use editable lists of populated wells in the configuration cell below. Choose only examples that suit your samples and installed filters. The cancellation example also starts a measurement.

In [ ]:
run_measurements = input("Run optional plate measurements? [y/N] ").strip().lower() in ("y", "yes")
measurement_examples = {
  "endpoint": "Full-plate endpoint absorbance at 450 nm",
  "partial": "Selected-well absorbance at 450 nm",
  "reference": "Reference-subtracted absorbance at 450 nm with a 620 nm reference",
  "kinetic": "Kinetic absorbance at 450 nm (3 reads, 15-second intervals)",
  "scan": "Absorbance well scan at 450 nm",
  "luminescence": "Luminescence (requires suitable luminescent samples)",
  "cancel": "Cancellation of a kinetic absorbance read at 450 nm",
}
selected_examples = set()
plate_loaded = False
if run_measurements:
  for key, label in measurement_examples.items():
    if input(f"Include {label}? [y/N] ").strip().lower() in ("y", "yes"):
      selected_examples.add(key)
if not selected_examples:
  print("No measurements selected. Measurement and plate-loading cells will be skipped.")

### Open the tray for the measurement plate

If at least one example was selected, clear the tray path and run this cell to open the tray. The next cell waits for you to load a plate; the tray will not close automatically while you are loading it.

In [ ]:
plate_loaded = False
if selected_examples:
  await reader.move_plate_tray_out()
else:
  print("Skipping measurement tray opening: no measurements selected.")

### Insert and confirm the plate

Place a compatible plate with samples or blanks in the intended wells into the tray, fully seated in the correct orientation. The geometry example below assumes a Costar 96-well clear plate in landscape orientation; for a different plate, update that geometry before running any read.

Remove your hands and clear the tray path, then enter `yes`. Enter or any other response skips all measurements and leaves the tray open. To try loading again, rerun the measurement tray-opening and confirmation cells before continuing.

In [ ]:
plate_loaded = False
if selected_examples:
  plate_loaded = input(
    "Is the measurement plate loaded and seated, with hands and tray path clear? [y/N] "
  ).strip().lower() in ("y", "yes")
  if not plate_loaded:
    print("Plate loading not confirmed. All measurements will be skipped; tray remains open.")
else:
  print("Skipping plate loading: no measurements selected.")

### Close the loaded tray

Close the tray only after the plate-loading confirmation above.

In [ ]:
if plate_loaded:
  await reader.move_plate_tray_in()
else:
  print("Skipping measurement tray closing: no confirmed plate load.")

## Plate geometry

The captured Costar 96-well clear plate geometry is available as a convenience. It must match the loaded plate, including orientation. For another plate, replace the assignment below with `PlateGeometry` using its measured dimensions in millimetres before proceeding. If you replace the plate between examples, update this geometry to match.

In [ ]:
from pylabrobot.molecular_devices.filtermax import PlateGeometry

plate = None
if plate_loaded:
  plate = PlateGeometry.costar_96_clear_landscape()
else:
  print("Skipping plate geometry: no confirmed measurement plate.")

### Configure populated wells

Edit the lists below for the selected examples, then run this cell. For example, set `"partial": ["C4", "C5", "D4"]` only if those wells contain suitable samples or blanks. Leave unselected examples as `[]`. Full-plate endpoint absorbance reads every well and does not use these lists.

There is no text-entry prompt for wells. An empty or invalid list for a selected example stops that cell before a read starts. Correct the relevant list here, rerun this configuration cell, then rerun the measurement cell. Lowercase names are accepted and duplicate wells are read once.

To change samples between examples, update the relevant list and rerun this cell. If an earlier input prompt is still waiting, use the notebook toolbar's **Interrupt/Stop** control before continuing.

In [ ]:
measurement_wells = {
  "partial": [],  # Example: ["C4", "C5", "D4"] — replace with your populated wells.
  "reference": [],
  "kinetic": [],
  "scan": [],
  "luminescence": [],
  "cancel": [],
}

### Readiness confirmations and well validation

Each selected example asks you to confirm that the correct plate is seated, its geometry and filters match, and your hands are clear. Enter skips that example. Reads eject the tray when finished, so you can replace the plate before the next example; the next read closes the tray after your confirmation.

The helpers below confirm readiness and validate the configured well lists. Defining them does not request input or operate the reader. Well validation uses the current plate geometry and never prompts for input.

In [ ]:
def confirm_measurement(example: str, requirements: str) -> bool:
  """Confirm readiness for a selected example with a previously loaded plate."""
  label = measurement_examples[example]
  if example not in selected_examples or not plate_loaded:
    print(f"Skipping {label}: not selected or plate loading not confirmed.")
    return False
  print(requirements)
  ready = input(
    f"Run {label}? Confirm plate seated, matching geometry/filters, and hands clear [y/N]: "
  ).strip().lower() in ("y", "yes")
  if not ready:
    print(f"Skipping {label}: readiness not confirmed.")
  return ready


def validate_wells(geometry: PlateGeometry, entries: list[str]) -> list[str]:
  """Validate configured populated wells without requesting interactive input."""
  valid_wells = {
    f"{chr(ord('A') + row)}{column + 1}"
    for row in range(geometry.rows)
    for column in range(geometry.columns)
  }
  wells = [well.strip().upper() for well in entries]
  if not wells or any(well not in valid_wells for well in wells):
    raise ValueError(
      f"Set measurement_wells for this example to a nonempty list of populated wells "
      f'within the {geometry.rows}x{geometry.columns} plate, e.g. ["C4", "C5"]. '
      "Rerun the configuration cell, then this measurement cell. No read was started."
    )
  return list(dict.fromkeys(wells))

## Absorbance

Optionally run a full-plate endpoint read at 450 nm: **every well is measured**, including empty wells. Choose the selected-well example instead if you only want to measure populated wells. Results are plate-shaped optical-density values. The tray ejects when the read finishes.

In [ ]:
if confirm_measurement(
  "endpoint",
  "This reads EVERY well at 450 nm. Confirm a suitable absorbance plate and a matching 450 nm excitation filter.",
):
  absorbance = await reader.read_absorbance(plate, wavelength=450)

Select populated wells for an optional partial-plate read at 450 nm. Unselected wells are `None` in the result.

In [ ]:
if confirm_measurement(
  "partial",
  "Use samples or blanks suitable for absorbance at 450 nm and a matching excitation filter.",
):
  wells = validate_wells(plate, measurement_wells["partial"])
  partial = await reader.read_absorbance(plate, wavelength=450, wells=wells)

For optional reference-subtracted absorbance, use supported 450 nm and 620 nm excitation filters matching the catalog and installed slide. Configure only the populated wells you want to read.

In [ ]:
if confirm_measurement(
  "reference",
  "Use absorbance samples or blanks and supported 450 nm and 620 nm excitation filters matching the catalog.",
):
  wells = validate_wells(plate, measurement_wells["reference"])
  reference_subtracted = await reader.read_absorbance(
    plate,
    wavelength=450,
    reference_wavelength=620,
    wells=wells,
  )

## Kinetic reads

Kinetic timing uses seconds. The result list contains one plate-shaped result per timepoint.

In [ ]:
if confirm_measurement(
  "kinetic",
  "Use absorbance samples or blanks and a matching 450 nm excitation filter. This collects 3 reads at 15-second intervals.",
):
  from pylabrobot.molecular_devices.filtermax import KineticTiming

  wells = validate_wells(plate, measurement_wells["kinetic"])
  kinetic = await reader.read_absorbance(
    plate,
    wavelength=450,
    wells=wells,
    kinetic=KineticTiming(interval=15, reads=3),
  )

## Well scans

The captured F5 software exposes horizontal and circular fill scans. Scan results preserve every `(x, y, value)` point for each selected well.

In [ ]:
if confirm_measurement(
  "scan",
  "Use absorbance samples or blanks and a matching 450 nm excitation filter for a circular fill scan.",
):
  from pylabrobot.molecular_devices.filtermax import WellScanSettings

  wells = validate_wells(plate, measurement_wells["scan"])
  scan = await reader.read_absorbance(
    plate,
    wavelength=450,
    wells=wells,
    well_scan=WellScanSettings(pattern="fill", density=5),
  )

## Luminescence

Read the open luminescence position on the installed emission slide. Values are raw RLU. Set `channels=2` for the captured dual-luminescence operation.

In [ ]:
if confirm_measurement(
  "luminescence",
  "Use a plate and samples suitable for luminescence, with the catalog-matched open position on the installed emission slide.",
):
  wells = validate_wells(plate, measurement_wells["luminescence"])
  luminescence = await reader.read_luminescence(
    plate,
    wells=wells,
    integration=0.4,
    read_height=1.0,
  )

## Cancel a read

`cancel_read()` is state-aware: it does nothing while idle and cooperatively stops an active read. A cancelled read raises `FilterMaxReadCancelled` in the read task and returns the reader to an idle, tray-ejected state.

In [ ]:
if confirm_measurement(
  "cancel",
  "This starts and then cancels a kinetic absorbance measurement. Use absorbance samples or blanks and a matching 450 nm excitation filter.",
):
  import asyncio

  from pylabrobot.molecular_devices.filtermax import FilterMaxReadCancelled, KineticTiming

  wells = validate_wells(plate, measurement_wells["cancel"])
  read_task = asyncio.create_task(
    reader.read_absorbance(
      plate,
      wavelength=450,
      wells=wells,
      kinetic=KineticTiming(interval=30, reads=10),
    )
  )
  await asyncio.sleep(2)
  await reader.cancel_read()
  try:
    await read_task
  except FilterMaxReadCancelled:
    pass

## Device errors

Inspect the exact firmware errors observed during this PLR session, then clear the local session log when handled.

In [ ]:
errors = await reader.get_error_log()
await reader.clear_error_log()

## Disconnect

Always release the serial port when finished.

In [ ]:
await reader.stop()